# Trainfer learning ResNet50 Chapter 10 Workshop 3

## Load module

In [21]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions

from tensorflow.keras.preprocessing.image import ImageDataGenerator


## Declare variable

In [22]:
target_img_shape = (224, 224)

train_dir = r'D:\Project\Learning\TensorFlow_Files\datasets\train_set'
test_dir = r'D:\Project\Learning\TensorFlow_Files\datasets\test_set'
val_dir = r'D:\Project\Learning\TensorFlow_Files\datasets\val_set'


train_datagen = ImageDataGenerator(
                            preprocessing_function=preprocess_input,
                            rotation_range=20,
                            width_shift_range=0.15,
                            height_shift_range=0.15,
                            shear_range=0.2,
                            zoom_range=0.2,
                            horizontal_flip=True,
                            fill_mode='nearest'
)

train_set = train_datagen.flow_from_directory(train_dir,
                                                  target_size=target_img_shape,
                                                  batch_size=32,
                                                  class_mode='binary')


val_datagen = ImageDataGenerator(
                            preprocessing_function=preprocess_input,
                            rotation_range=20,
                            width_shift_range=0.15,
                            height_shift_range=0.15,
                            shear_range=0.2,
                            zoom_range=0.2,
                            horizontal_flip=True,
                            fill_mode='nearest'
)

val_set = val_datagen.flow_from_directory(val_dir,
                                                  target_size=target_img_shape,
                                                  batch_size=32,
                                                  class_mode='binary')

Found 15997 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


## Check data_set

In [23]:
batch = next(train_set) # train_set.next()
# train_set.next()
print(batch[0].shape)
print(batch[1].shape)

(32, 224, 224, 3)
(32,)


## Create base model from resnet50 model

In [24]:
in_shape = (target_img_shape[0], target_img_shape[1], 3)

In [25]:
base_model = ResNet50(weights='imagenet', 
                      include_top=False, 
                      input_shape=in_shape)

base_model.summary()

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_7 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_7[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

## Create tranfer learning model

In [26]:
tranfer_resnet50_model = Sequential()

tranfer_resnet50_model.add(base_model)
tranfer_resnet50_model.add(Flatten())
tranfer_resnet50_model.add(Dense(128, activation='relu'))
tranfer_resnet50_model.add(Dense(1, activation='sigmoid'))

tranfer_resnet50_model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 flatten_4 (Flatten)         (None, 100352)            0         
                                                                 
 dense_8 (Dense)             (None, 128)               12845184  
                                                                 
 dense_9 (Dense)             (None, 1)                 129       
                                                                 
Total params: 36,433,025
Trainable params: 36,379,905
Non-trainable params: 53,120
_________________________________________________________________


## Freeze base model

In [ ]:
base_model.trainable = False

## Compile tranfer model

In [27]:
tranfer_resnet50_model.compile(optimizer='adam',
                              loss='binary_crossentropy',
                              metrics=['accuracy'])

## Train tranfer model

In [28]:
es = EarlyStopping(monitor='val_loss',
                   patience=5,
                   verbose=1,
                   restore_best_weights=True)
mc = ModelCheckpoint('model/tranfer_resnet50_model.h5',
                       monitor='val_accuracy',
                       save_best_only=True,
                       verbose=1)

history = tranfer_resnet50_model.fit(train_set,
                                    steps_per_epoch=len(train_set),
                                    validation_data=val_set,
                                    validation_steps=len(val_set),
                                    callbacks=[es, mc],
                                    epochs=10,
                                    verbose=1)

Epoch 1/10
500/500 [==============================] - ETA: 0s - loss: 0.6333 - accuracy: 0.7916
Epoch 1: val_accuracy improved from -inf to 0.79050, saving model to model\tranfer_resnet50_model.h5
500/500 [==============================] - 136s 262ms/step - loss: 0.6333 - accuracy: 0.7916 - val_loss: 1.0282 - val_accuracy: 0.7905
Epoch 2/10
500/500 [==============================] - ETA: 0s - loss: 0.2245 - accuracy: 0.9049
Epoch 2: val_accuracy improved from 0.79050 to 0.86550, saving model to model\tranfer_resnet50_model.h5
500/500 [==============================] - 130s 260ms/step - loss: 0.2245 - accuracy: 0.9049 - val_loss: 0.3410 - val_accuracy: 0.8655
Epoch 3/10
500/500 [==============================] - ETA: 0s - loss: 0.1754 - accuracy: 0.9286
Epoch 3: val_accuracy improved from 0.86550 to 0.91000, saving model to model\tranfer_resnet50_model.h5
500/500 [==============================] - 130s 259ms/step - loss: 0.1754 - accuracy: 0.9286 - val_loss: 0.2455 - val_accuracy: 0.910